In [1]:
from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from chat_memory import get_chat_history
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage
from video_processing import analyze_video
from vectorstore_setup import clean_classification_text
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="path/to/your/chroma_db", embedding_function=embeddings)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")



# when a query comes in, use this model to embed the query text so you can compare it against the vectors already stored.
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/chroma_db", 
                                    embedding_function=embeddings)


# Whisper speach to text 
from openai import OpenAI
client = OpenAI()




In [2]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Your job is to classify user queries into two buckets: 
     - memory
     - memory and vectorstore
     
     *Guidelines*
     Questions regarding exercises, exercise form, or technique: vectorstore & memory
     General question or statements: memory
   
     *Respond to queries with only the correct class*
     Examples: 
     User: how should I grip the bar for bench press?
     Agent: vectorstore & memory

     User: what muscles should I feel during the Romanian dead lift?
     Agent: vectorstore & memory

     User: How's this? 
     Agent: vectorstore & memory

     User: Is this good squat technique? 
     Agent: vectorstore & memory

     User: Was my bar path better?
     Agent: vectorstore & memory

     User: thakns for your help!
     Agent: memory

"""),

     ("human", "{query}")
     
])

llm = ChatOpenAI(model='gpt-4o')

output_parser = StrOutputParser()

router_chain = router_prompt | llm | output_parser


In [3]:
# Define the state

# the state is a dictionary

class GraphState(TypedDict):

    session_id: int 

    user_query: str

    user_video: str

    classification_image: str

    classified_keywords: str

    top_k_chunks: str

    encoded_images: list[str]

    response: str




In [4]:
workflow = StateGraph(GraphState) 

In [5]:
def video_encoder_node(state: GraphState):
    user_video = state["user_video"]
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=15, max_seconds=10) # encodes video
    encoded_images = [] # creates a list of encoded images for LLM
    for images in user_video_encoded:
        encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
        classification_image = user_video_encoded[0] 
    
    return {"classification_image": classification_image, "encoded_images": encoded_images}


In [6]:
# video classification router NODE

def video_classification_node(state: GraphState):
    classification_image = state["classification_image"]

    router_llm = ChatOpenAI(model='gpt-4o')

    response = router_llm.invoke([

    {"role": "user", "content": 

    [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
    Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
    Output format should be exactly the example below.
    **Example**
    Overhead press

    1. Feet & base
    2. Glutes & legs
    3. Core & Ribcage
    4. Shoulder position
    5. Bar path
    6. Head & Neck
    7. Lockout position
    8. Tempo and control
    """},


        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
        ]}


    ])
    print(response.text)

    classified_keywords = clean_classification_text(response)

    return {"classified_keywords": classified_keywords}


In [7]:
def vector_db_node(state: GraphState):
    user_query = state["user_query"]
    classified_keywords = state.get("classified_keywords", "")
    retrieval_query = classified_keywords

    if classified_keywords:
        results_user = vectorstore.similarity_search(user_query, k=5)
        results_video = vectorstore.similarity_search(retrieval_query, k=5)
        results = results_user + results_video

    if not classified_keywords: 
        results_user = vectorstore.similarity_search(user_query, k=5)
        results = results_user

    unique_results = []
    seen = set()

    for r in results:
        content = r.page_content
        if content not in seen:
            seen.add(content)
            unique_results.append(r)


    top_k_chunks = "\n".join([r.page_content for r in unique_results])

    return {"top_k_chunks": top_k_chunks}

In [8]:
def response_generator(state: GraphState):
    top_k_chunks = state['top_k_chunks']
    encoded_images = state.get("encoded_images", [])
    session_id = state["session_id"]
    user_query = state["user_query"]

    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and arefuly for problems or issues related to best practices in exercise form. Help the user diagnose their incorrect form. 
    Be specific about what you observe.

    # ANSWER CONTEXT
    Use ONLY the following context when answering a user: 
        
    ---   
    {top_k_chunks}
    ---
    if the query or image isn't in context, reply, 'I don't have expert coaching content for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])



    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,   # <- your list of {"type":"image_url",...}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg], "top_k_chunks": top_k_chunks},
        config={"configurable": {"session_id": session_id}}
    )


    return {"response": response}
    

In [9]:
def chat_memory(state: GraphState):
    user_query = state["user_query"]
    session_id = state["session_id"]
    
    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg]},
        config={"configurable": {"session_id": session_id}}
    )

    return {"response": response}
    


# Router function

In [10]:
def route_query(state: GraphState):
    # first check: is there a video?
    if state["user_video"]:
        return "video_encoder"
    
    # no video — ask the LLM which path
    route = router_chain.invoke({"query": state["user_query"]})
    route = route.lower().strip()
    
    if "vectorstore" in route:
        return "vector_db"
    else:
        return "chat_memory"

# Audio transcription function

In [11]:
def transcribe_audio(audio_path):
    audio_file = open(audio_path, "rb") # rb = read binary. audio files are binary data (raw bytes). this tells python gto open as binary instaed of text
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

    return transcription.text
    

user_query = transcribe_audio("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_voice_notes/bench_press_voice_note.m4a")
print(user_query)

Why does it help to keep my butt on the bench once the bar is out?


In [12]:
# conditional edge
workflow.add_conditional_edges(START, route_query)

# add nodes
workflow.add_node("video_encoder", video_encoder_node)
workflow.add_node("video_classification_router", video_classification_node)
workflow.add_node("vector_db", vector_db_node)
workflow.add_node("response_generator", response_generator)
workflow.add_node("chat_memory", chat_memory)



# connect them
workflow.add_edge("chat_memory", END)
workflow.add_edge("video_encoder", "video_classification_router")
workflow.add_edge("video_classification_router", "vector_db")
workflow.add_edge("vector_db", "response_generator")
workflow.add_edge("response_generator", END)

app = workflow.compile()


In [13]:
result = app.invoke({
    "session_id": 123,
    "user_query": "how's my form?",
    "user_video": "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/user_01_bench.MP4"
})

print(result["response"])

Frames processed: 15, (10s cap)
Saved to processed-images/user_01_bench
Video name: user_01_bench
Bench Press

1. Feet & base
2. Glutes & legs
3. Arch & lower back
4. Shoulder position
5. Hand grip & bar placement
6. Head & Neck
7. Bar path
8. Lockout position
9. Tempo and control
Great arch and upper‑back tightness—you’re clearly setting up to use your back as a platform. A few fixes will make this much stronger and safer:

- Glutes must stay down. In several frames your butt comes off the pad. You need four points of contact the whole set: head, upper back, glutes, and feet. After the liftoff, drop your hips so your butt touches the bench and keep it there. Use leg drive to reinforce the arch, not to bridge.

- Foot/leg drive. You’re up on your toes. For the most efficient drive (and for regulation technique), plant your heels on the floor. Set your feet so your shins are nearly vertical and keep your legs close to the bench when viewed from the front. You may need to point your toes

In [ ]:

result = app.invoke({
    "session_id": 123,
    "user_query": user_query,
    "user_video": ""
})

print(result["response"])

In [15]:
result = app.invoke({
    "session_id": 123,
    "user_query": "thanks for your help today!",
    "user_video": ""
})

print(result["response"])

You’re welcome! Quick pre‑set checklist to lock it in next time:

- Eyes under the bar, shoulder blades pinched, big breath and brace.
- Feet set wide enough to keep heels down; squeeze glutes so your butt stays in contact.
- Touch lower chest, press back in a slight J path while driving heels into the floor.

Drills to groove it:
- 2–3 sec paused bench (hips glued to pad).
- Spoto press (hover 1–2 cm above chest) to keep tension and bar path.
- Empty‑bar liftoff practice: unrack, set butt down, then do 3–5 crisp singles.

If you want, send a short side or foot‑end video next session and I’ll fine‑tune bar path and elbow angle.
